# Position Sizing in QuantEval
# QuantEval 中的仓位管理

By default, every strategy in QuantEval uses an **all-in / all-out** approach: when the signal is 1 the entire portfolio is deployed, and when the signal is 0 it moves to cash. While simple, this approach allocates capital in a binary way that ignores portfolio risk.

默认情况下，QuantEval 中的所有策略采用**全仓进出**方式：信号为 1 时全仓入场，信号为 0 时全部持现。这种方式简单直接，但忽略了资金风险管理。

**Position sizing** solves this by converting a binary signal into a fractional exposure — for example, investing only 50% of capital, or scaling size down when volatility is high.

**仓位管理**通过将二值信号转换为分数仓位来解决这一问题——例如每次只投入 50% 的资金，或在高波动率时缩小仓位。

QuantEval provides four built-in `PositionSizer` classes:

QuantEval 提供了四种内置的 `PositionSizer` 类：

| Sizer | Description (EN) | 说明 (中文) |
|---|---|---|
| `AllInSizer` | All-in / all-out (default) | 全仓进出（默认） |
| `FixedFractionSizer(fraction)` | Always invest a fixed % | 固定比例投入 |
| `EqualWeightSizer(n_positions)` | Divide capital by N | 等权分配给 N 个仓位 |
| `VolatilityTargetSizer(target_vol, window)` | Scale inversely to volatility | 与波动率成反比调整仓位 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quanteval import (
    Backtester,
    AllInSizer,
    FixedFractionSizer,
    EqualWeightSizer,
    VolatilityTargetSizer,
    PositionSizer,
)
from quanteval.strategies import DualMAStrategy

rng = np.random.default_rng(42)
index = pd.date_range('2020-01-02', periods=500, freq='B')
log_returns = rng.normal(0.0003, 0.012, len(index))
close = pd.Series(100.0 * np.exp(np.cumsum(log_returns)), index=index)

data = pd.DataFrame({
    'Open':   close * 0.998,
    'High':   close * 1.010,
    'Low':    close * 0.990,
    'Close':  close,
    'Volume': rng.integers(1_000, 5_000, len(index)).astype(float),
    'Amount': close * rng.integers(1_000, 5_000, len(index)),
}, index=index)
data['Ret'] = data['Close'].pct_change().fillna(0.0)

strategy = DualMAStrategy(fast_window=10, slow_window=40)
print(f'Data range: {index[0].date()} to {index[-1].date()}, {len(data)} bars')

## 1. AllInSizer — Default Behaviour
## 1. AllInSizer — 默认行为（全仓进出）

`AllInSizer` is the default. When the signal is 1, 100% of capital is deployed. This is identical to running `Backtester` without any `sizer` argument.

`AllInSizer` 是默认配置。信号为 1 时投入 100% 资金，与不传 `sizer` 参数时行为完全一致。

In [ ]:
result_all_in = Backtester(
    strategy=strategy,
    data=data,
    transaction_costs=False,
    sizer=AllInSizer(),
).run()

print('AllInSizer — unique position values:', sorted(result_all_in.positions.unique()))
result_all_in.summary()[['total_return', 'annual_return', 'sharpe_ratio', 'max_drawdown']]

## 2. FixedFractionSizer — Invest a Fixed Percentage
## 2. FixedFractionSizer — 固定比例投入

`FixedFractionSizer(fraction=0.5)` invests 50% of the portfolio whenever the strategy signals long, keeping the other 50% in cash at all times. This reduces both returns and drawdowns proportionally.

`FixedFractionSizer(fraction=0.5)` 在策略发出多头信号时仅投入 50% 的资金，另外 50% 始终保持现金。这会等比例地降低收益和回撤。

In [ ]:
result_half = Backtester(
    strategy=strategy,
    data=data,
    transaction_costs=False,
    sizer=FixedFractionSizer(fraction=0.5),
).run()

print('FixedFractionSizer(0.5) — unique position values:', sorted(result_half.positions.unique()))
result_half.summary()[['total_return', 'annual_return', 'sharpe_ratio', 'max_drawdown']]

## 3. EqualWeightSizer — Divide Capital Across N Positions
## 3. EqualWeightSizer — 等权分配给 N 个仓位

`EqualWeightSizer(n_positions=5)` sizes each position as 1/5 = 20% of portfolio. This is useful when you run the same sizer across multiple instruments and want equal capital allocation to each.

`EqualWeightSizer(n_positions=5)` 将每个仓位的资金设为组合净值的 1/5 = 20%。适用于同时运行多个品种策略并希望等权分配资金的场景。

In [ ]:
result_eq = Backtester(
    strategy=strategy,
    data=data,
    transaction_costs=False,
    sizer=EqualWeightSizer(n_positions=5),
).run()

print('EqualWeightSizer(5) — unique position values:', sorted(result_eq.positions.unique()))
result_eq.summary()[['total_return', 'annual_return', 'sharpe_ratio', 'max_drawdown']]

## 4. VolatilityTargetSizer — Size Inversely to Volatility
## 4. VolatilityTargetSizer — 根据波动率动态调整仓位

`VolatilityTargetSizer(target_vol=0.15, window=20)` targets a **15% annualised volatility contribution**. When recent volatility is high, the position is scaled down; when volatility is low, the position is scaled up (capped at 1.0 — no leverage).

Position fraction = `min(target_vol / rolling_std, 1.0)`

`VolatilityTargetSizer(target_vol=0.15, window=20)` 以 **15% 年化波动率贡献**为目标。当近期波动率高时缩小仓位，当波动率低时放大仓位（最高 1.0，不允许杠杆）。

仓位比例 = `min(目标波动率 / 滚动波动率, 1.0)`

In [ ]:
result_vol = Backtester(
    strategy=strategy,
    data=data,
    transaction_costs=False,
    sizer=VolatilityTargetSizer(target_vol=0.15, window=20),
).run()

print(f'VolatilityTargetSizer — position range: [{result_vol.positions.min():.3f}, {result_vol.positions.max():.3f}]')
result_vol.summary()[['total_return', 'annual_return', 'sharpe_ratio', 'max_drawdown']]

## 5. Comparing All Four Sizers
## 5. 四种仓位计算器对比

The chart below overlays the normalised equity curves for all four sizers on the same strategy and data.

下图展示了四种仓位计算器在相同策略和数据下的归一化净值曲线。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1]})

results = {
    'AllInSizer (100%)': result_all_in,
    'FixedFraction (50%)': result_half,
    'EqualWeight (1/5)': result_eq,
    'VolatilityTarget (15%)': result_vol,
}

ax = axes[0]
for label, res in results.items():
    ax.plot(res.equity_curve.index, res.equity_curve, label=label, linewidth=1.5)

ax.set_title('Equity Curves — Position Sizing Comparison\n净值曲线对比', fontsize=13)
ax.set_ylabel('Equity (normalised to 1.0) / 归一化净值')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(result_vol.positions.index, result_vol.positions, color='steelblue', linewidth=0.8, alpha=0.8)
ax2.set_title('VolatilityTargetSizer — Dynamic Position Size / 波动率目标仓位动态变化', fontsize=10)
ax2.set_ylabel('Position fraction / 仓位比例')
ax2.set_ylim(-0.05, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

metrics = pd.DataFrame({
    label: res.summary()[['total_return', 'annual_return', 'annual_volatility', 'sharpe_ratio', 'max_drawdown']]
    for label, res in results.items()
}).T
metrics.columns = ['Total Return', 'Annual Return', 'Annual Vol', 'Sharpe', 'Max Drawdown']
metrics.style.format({
    'Total Return': '{:.2%}',
    'Annual Return': '{:.2%}',
    'Annual Vol': '{:.2%}',
    'Sharpe': '{:.3f}',
    'Max Drawdown': '{:.2%}',
})

## 6. Writing a Custom PositionSizer
## 6. 自定义仓位计算器

You can create your own sizer by inheriting from `PositionSizer` and implementing the `apply` method.

通过继承 `PositionSizer` 并实现 `apply` 方法，可以创建自定义仓位计算器。

The example below sizes down to 30% after a new position is entered (mimicking a cautious initial entry), then scales back up to full after 5 bars of holding.

下面的示例在新建仓位后将仓位压缩至 30%（模拟谨慎建仓），持有 5 个交易日后逐步恢复全仓。

In [ ]:


class RampInSizer(PositionSizer):
    """Start at initial_fraction on entry, ramp up to 1.0 over ramp_bars.
    建仓时以 initial_fraction 入场，在 ramp_bars 个交易日内线性增至全仓。"""

    def __init__(self, initial_fraction: float = 0.3, ramp_bars: int = 5) -> None:
        self.initial_fraction = initial_fraction
        self.ramp_bars = ramp_bars

    def apply(self, raw_positions: pd.Series, data: pd.DataFrame) -> pd.Series:
        result = raw_positions.copy()
        in_trade = False
        bars_held = 0
        for i, (idx, pos) in enumerate(raw_positions.items()):
            if pos == 0.0:
                in_trade = False
                bars_held = 0
            else:
                if not in_trade:
                    in_trade = True
                    bars_held = 0
                bars_held += 1
                ramp_frac = self.initial_fraction + (1.0 - self.initial_fraction) * min(
                    bars_held / self.ramp_bars, 1.0
                )
                result.iloc[i] = ramp_frac
        return result


result_ramp = Backtester(
    strategy=strategy,
    data=data,
    transaction_costs=False,
    sizer=RampInSizer(initial_fraction=0.3, ramp_bars=5),
).run()

print(f'RampInSizer — position range: [{result_ramp.positions.min():.3f}, {result_ramp.positions.max():.3f}]')
result_ramp.summary()[['total_return', 'annual_return', 'sharpe_ratio', 'max_drawdown']]

## Summary
## 总结

| Concept | How to use it |
|---|---|
| Default (all-in) | `Backtester(strategy, data)` — no `sizer` argument needed |
| Fixed fraction | `sizer=FixedFractionSizer(fraction=0.5)` |
| Equal weight across N | `sizer=EqualWeightSizer(n_positions=5)` |
| Volatility targeting | `sizer=VolatilityTargetSizer(target_vol=0.15, window=20)` |
| Custom logic | Subclass `PositionSizer`, implement `apply(raw_positions, data)` |

Key points / 要点：
- Strategies always return **binary `{0, 1}` signals** — position sizing is a separate concern applied by the `Backtester`. 策略始终返回**二值 `{0, 1}` 信号**，仓位管理由 `Backtester` 独立应用。
- All sizers return positions in `[0.0, 1.0]` — no leverage. 所有仓位计算器均返回 `[0.0, 1.0]` 范围内的仓位，不允许杠杆。
- Transaction costs are applied **after** position sizing, so fractional sizing naturally reduces costs proportionally. 交易成本在仓位计算**之后**应用，因此分数仓位会自然等比例降低交易成本。